# 🏠 House Price Prediction – Exploratory Data Analysis (EDA)

This notebook walks through the dataset inspection, cleaning summary, distribution analysis,
correlation study, and feature-vs-price relationships that informed our feature engineering choices.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

plt.rcParams.update({
    'figure.facecolor': '#0f172a',
    'axes.facecolor':   '#1e293b',
    'axes.edgecolor':   '#475569',
    'axes.labelcolor':  '#cbd5e1',
    'xtick.color':      '#94a3b8',
    'ytick.color':      '#94a3b8',
    'text.color':       '#e2e8f0',
    'grid.color':       '#334155',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
})
sns.set_style('darkgrid')
ACCENT = '#6366f1'

## 1. Load the Dataset

In [ ]:
df = pd.read_csv('../data/house_prices.csv')
print(f'Shape: {df.shape}  →  {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
print('=== Data Types & Non-Null Counts ===')
df.info()
print()
print('=== Basic Statistics ===')
df.describe(include='all').T

## 2. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]
print('Columns with missing values:')
print(missing_df.to_string())

# Heatmap
fig, ax = plt.subplots(figsize=(12, 3))
sns.heatmap(df.isnull().T, cbar=False, cmap='RdYlGn_r', ax=ax, linewidths=0.5)
ax.set_title('Missing Value Heatmap (yellow = missing)', fontsize=14, pad=10)
ax.set_xlabel('Row Index')
plt.tight_layout()
plt.show()

## 3. Price Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(df['Price'].dropna() / 1e6, bins=35, color=ACCENT, edgecolor='#0f172a', alpha=0.9)
axes[0].set_title('Price Distribution', fontsize=13)
axes[0].set_xlabel('Price (₹ Millions)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'₹{x:.0f}M'))

# Log-scale histogram
axes[1].hist(np.log1p(df['Price'].dropna()), bins=35, color='#06b6d4', edgecolor='#0f172a', alpha=0.9)
axes[1].set_title('Log(Price + 1) Distribution', fontsize=13)
axes[1].set_xlabel('log₁(Price)')
axes[1].set_ylabel('Count')

plt.suptitle('House Price Distribution', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f'Mean  : ₹{df["Price"].mean():,.0f}')
print(f'Median: ₹{df["Price"].median():,.0f}')
print(f'Std   : ₹{df["Price"].std():,.0f}')
print(f'Min   : ₹{df["Price"].min():,.0f}')
print(f'Max   : ₹{df["Price"].max():,.0f}')

## 4. Categorical Feature Distributions

In [ ]:
cat_cols = ['Location', 'Condition']
fig, axes = plt.subplots(1, len(cat_cols), figsize=(13, 5))

palette = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b']

for ax, col in zip(axes, cat_cols):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values, color=palette[:len(counts)], edgecolor='#0f172a', alpha=0.9)
    ax.set_title(f'{col} Distribution', fontsize=13)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
                ha='center', va='bottom', fontsize=10, color='#e2e8f0')

plt.suptitle('Categorical Feature Counts', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Numeric Feature Distributions

In [ ]:
num_cols = ['Area', 'Bedrooms', 'Bathrooms', 'Floors', 'Garage', 'Age']
num_cols = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
colors = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b', '#ec4899', '#f97316']

for i, (col, color) in enumerate(zip(num_cols, colors)):
    axes[i].hist(df[col].dropna(), bins=25, color=color, edgecolor='#0f172a', alpha=0.85)
    axes[i].set_title(col, fontsize=12)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')

for j in range(len(num_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Numeric Feature Distributions', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['Year Built'], errors='ignore')
corr = numeric_df.corr()

mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(11, 8))
cmap = sns.diverging_palette(250, 15, as_cmap=True)
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap=cmap,
    vmin=-1, vmax=1, center=0,
    linewidths=0.6, linecolor='#0f172a',
    ax=ax, annot_kws={'size': 9},
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('\nTop correlations with Price:')
print(corr['Price'].drop('Price').abs().sort_values(ascending=False).to_string())

## 7. Feature vs Price Scatter Plots

In [ ]:
scatter_cols = ['Area', 'Bedrooms', 'Bathrooms', 'Age', 'Garage']
scatter_cols = [c for c in scatter_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
scatter_colors = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b', '#ec4899']

for i, (col, color) in enumerate(zip(scatter_cols, scatter_colors)):
    clean = df[[col, 'Price']].dropna()
    axes[i].scatter(clean[col], clean['Price'] / 1e6, alpha=0.5, color=color, s=18, edgecolors='none')
    z = np.polyfit(clean[col].astype(float), clean['Price'].astype(float) / 1e6, 1)
    p = np.poly1d(z)
    x_line = np.linspace(clean[col].min(), clean[col].max(), 200)
    axes[i].plot(x_line, p(x_line), color='white', linewidth=1.5, linestyle='--', alpha=0.8, label='Trend')
    axes[i].set_title(f'{col} vs Price', fontsize=12)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Price (₹M)')
    axes[i].legend(fontsize=8)

for j in range(len(scatter_cols), len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Feature vs Price Relationships', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Price by Location (Box Plot)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Box plot
location_order = ['Urban', 'Semi Urban', 'Rural']
location_colors = ['#6366f1', '#06b6d4', '#10b981']
groups = [df.loc[df['Location'] == loc, 'Price'].dropna() / 1e6 for loc in location_order]
bp = axes[0].boxplot(groups, labels=location_order, patch_artist=True, notch=False,
                     medianprops=dict(color='white', linewidth=2))
for patch, color in zip(bp['boxes'], location_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
for whisker in bp['whiskers'] + bp['caps']:
    whisker.set_color('#94a3b8')
for flier in bp['fliers']:
    flier.set_markerfacecolor('#f87171')
    flier.set_markersize(4)
axes[0].set_title('Price Distribution by Location', fontsize=13)
axes[0].set_xlabel('Location')
axes[0].set_ylabel('Price (₹M)')

# Mean price bar
mean_prices = df.groupby('Location')['Price'].mean() / 1e6
mean_prices = mean_prices.reindex(location_order)
axes[1].bar(mean_prices.index, mean_prices.values, color=location_colors, edgecolor='#0f172a', alpha=0.9)
for i, val in enumerate(mean_prices.values):
    axes[1].text(i, val + 0.1, f'₹{val:.1f}M', ha='center', fontsize=11, color='#e2e8f0')
axes[1].set_title('Mean Price by Location', fontsize=13)
axes[1].set_xlabel('Location')
axes[1].set_ylabel('Mean Price (₹M)')

plt.suptitle('Price vs Location Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Price by Condition

In [ ]:
cond_order = ['Excellent', 'Good', 'Fair', 'Poor']
cond_colors = ['#10b981', '#6366f1', '#f59e0b', '#f87171']
mean_cond = df.groupby('Condition')['Price'].mean().reindex(cond_order) / 1e6

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(mean_cond.index, mean_cond.values, color=cond_colors, edgecolor='#0f172a', alpha=0.9)
for bar, val in zip(bars, mean_cond.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'₹{val:.1f}M',
            ha='center', va='bottom', fontsize=11, color='#e2e8f0')
ax.set_title('Mean Price by House Condition', fontsize=14, fontweight='bold')
ax.set_xlabel('Condition')
ax.set_ylabel('Mean Price (₹M)')
plt.tight_layout()
plt.show()

## 10. Key EDA Insights

In [ ]:
print('='*60)
print('          KEY EDA INSIGHTS')
print('='*60)
print(f'  Dataset: {len(df)} rows × {len(df.columns)} columns')
print(f'  Price range: ₹{df["Price"].min()/1e6:.1f}M – ₹{df["Price"].max()/1e6:.1f}M')
print(f'  Mean price : ₹{df["Price"].mean()/1e6:.2f}M')
print()
print('  Strongest Price Predictors (by correlation):')
corr_with_price = numeric_df.corr()['Price'].drop('Price').abs().sort_values(ascending=False)
for feat, val in corr_with_price.head(5).items():
    print(f'    {feat:<15} r = {val:.3f}')
print()
print('  Location Premium (vs Rural):')
base_price = df[df['Location']=='Rural']['Price'].mean()
for loc in ['Semi Urban', 'Urban']:
    diff = df[df['Location']==loc]['Price'].mean() - base_price
    print(f'    {loc:<12} +₹{diff/1e6:.2f}M')
print('='*60)